# Silver Layer - Data Cleaning (Unity Catalog Compatible)

**Source:** `workspace.rebu.bronze_*` tables  
**Target:** `workspace.rebu.silver_*` tables

This notebook transforms Bronze layer data into clean, standardized Silver layer tables.

---

## Notebook Summary

- Imports PySpark, Delta Lake, and helper functions for data cleaning.
- Configures Unity Catalog and schema for Bronze and Silver tables.
- Defines helper functions to reference Bronze and Silver table names.
- Implements a generic function to clean and save data from Bronze to Silver tables.
- Checks existence and record counts of all required Bronze tables.
- Defines transformation functions for each dimension table:
  - Taxi Type: Cleans fare, rate, capacity, and type name.
  - Taxi: Standardizes license plate, model, status, and year.
  - Drivers: Cleans names, phone, email, license, gender, age, experience, rating, join date.
  - Passengers: Cleans names, phone, email, category, gender, age, registration date, address (Singapore standardization).
  - Payment Type: Cleans payment type names.
  - Place: Cleans place names (Singapore standardization), latitude, longitude, postal code.
- Applies transformations and writes cleaned data to Silver tables.
- Displays sample records from each Silver table for verification.

## Setup and Configuration

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, trim, upper, lower, regexp_replace, 
    to_timestamp, to_date, current_timestamp, coalesce, lit
)
from pyspark.sql.types import StringType, DoubleType, IntegerType
from delta.tables import DeltaTable

# Unity Catalog Configuration
UC_CATALOG = "workspace"
UC_SCHEMA = "rebu"

print("="*80)
print("SILVER LAYER CONFIGURATION")
print("="*80)
print(f"Reading from: {UC_CATALOG}.{UC_SCHEMA}.bronze_*")
print(f"Writing to: {UC_CATALOG}.{UC_SCHEMA}.silver_*")
print("="*80)

## Helper Functions

- `bronze_table(name)`:  
  Returns the full Unity Catalog table name for a Bronze layer table, given its base name.

- `silver_table(name)`:  
  Returns the full Unity Catalog table name for a Silver layer table, given its base name.

- `clean_and_save_to_silver(bronze_name, silver_name, transformation_func=None)`:  
  Reads data from a Bronze table, applies optional cleaning/transformation, adds Silver layer metadata, and writes the result to the corresponding Silver table.

In [0]:
def bronze_table(name):
    """Get full Bronze table name"""
    return f"{UC_CATALOG}.{UC_SCHEMA}.bronze_{name}"

def silver_table(name):
    """Get full Silver table name"""
    return f"{UC_CATALOG}.{UC_SCHEMA}.silver_{name}"

In [0]:
def clean_and_save_to_silver(bronze_name, silver_name, transformation_func=None):
    """
    Read from Bronze, apply transformations, and save to Silver layer
    
    Args:
        bronze_name: Name of the Bronze table (e.g., "taxi_type")
        silver_name: Name of the Silver table (e.g., "taxi_type")
        transformation_func: Optional function to apply custom transformations
    """
    print(f"\n{'='*80}")
    print(f"Processing: {bronze_name} → {silver_name}")
    print(f"{'='*80}")
    
    # Read from Bronze with full UC name
    bronze_full = bronze_table(bronze_name)
    print(f"Reading from: {bronze_full}")
    
    try:
        df = spark.table(bronze_full)
    except Exception as e:
        print(f"Error reading {bronze_full}: {e}")
        print(f"\nTroubleshooting:")
        print(f"  1. Check table exists: SHOW TABLES IN {UC_CATALOG}.{UC_SCHEMA}")
        print(f"  2. Run Bronze layer notebook first")
        raise
    
    # Remove Bronze metadata columns
    bronze_metadata_cols = ["ingestion_timestamp", "source_file", "source_volume"]
    for col_name in bronze_metadata_cols:
        if col_name in df.columns:
            df = df.drop(col_name)
    
    # Apply custom transformation if provided
    if transformation_func:
        print(f"Applying transformations...")
        df = transformation_func(df)
    
    # Add Silver layer metadata
    df_with_metadata = df \
        .withColumn("silver_processed_timestamp", current_timestamp()) \
        .withColumn("data_quality_flag", lit("VALID"))
    
    # Write to Silver layer
    silver_full = silver_table(silver_name)
    print(f"Writing to: {silver_full}")
    
    df_with_metadata.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(silver_full)
    
    record_count = df_with_metadata.count()
    print(f"✓ SUCCESS: Processed {record_count:,} records")
    print(f"{'='*80}\n")
    
    return df_with_metadata

## Verify Bronze Tables Exist

In [0]:
print("\nChecking for Bronze tables...")
print("-"*80)

bronze_tables = [
    "taxi_type", "taxi", "drivers", "passengers", 
    "payment_type", "place", "booking", "rewards"
]

missing_tables = []
for table in bronze_tables:
    table_full = bronze_table(table)
    try:
        count = spark.table(table_full).count()
        print(f"- {table_full:<50} {count:>10,} records")
    except Exception as e:
        print(f"{table_full:<50} NOT FOUND")
        missing_tables.append(table)

if missing_tables:
    print("\n Some Bronze tables are missing!")
    print("   Please run the Bronze layer notebook first:")
    print("   01_Bronze_Layer_From_Volume.py")
    print(f"\n   Missing tables: {', '.join(missing_tables)}")
else:
    print("\nAll Bronze tables found!")

print("-"*80)

## Transform Dimension Tables

- **Taxi Type:** Cleans and standardizes taxi type data (fare, rate, capacity, type name).
- **Taxi:** Standardizes license plate, model, status, and year manufactured.
- **Drivers:** Cleans driver names, phone, email, license, gender, age, experience, rating, join date.
- **Passengers:** Cleans passenger names, phone, email, category, gender, age, registration date, and standardizes Singapore addresses.
- **Payment Type:** Cleans payment type names.
- **Place:** Cleans place names (Singapore standardization), latitude, longitude, and postal code.

### Taxi Type

In [0]:
def transform_taxi_type(df):
    """Clean TaxiType data"""
    return df \
        .withColumn("TypeName", trim(col("TypeName"))) \
        .withColumn("BaseFare", col("BaseFare").cast(DoubleType())) \
        .withColumn("PerKmRate", col("PerKmRate").cast(DoubleType())) \
        .withColumn("Capacity", col("Capacity").cast(IntegerType())) \
        .na.fill({
            "TypeName": "Unknown",
            "BaseFare": 0.0,
            "PerKmRate": 0.0,
            "Capacity": 4
        })

df_taxi_type_silver = clean_and_save_to_silver("taxi_type", "taxi_type", transform_taxi_type)
display(df_taxi_type_silver)

### Taxi

In [0]:
def transform_taxi(df):
    """Clean Taxi data"""
    return df \
        .withColumn("LicensePlate", upper(trim(col("LicensePlate")))) \
        .withColumn("Model", trim(col("Model"))) \
        .withColumn("Status", trim(col("Status"))) \
        .withColumn("YearManufactured", col("YearManufactured").cast(IntegerType())) \
        .na.fill({
            "LicensePlate": "UNKNOWN",
            "Model": "Unknown Model",
            "Status": "Unknown",
            "YearManufactured": 2020
        })

df_taxi_silver = clean_and_save_to_silver("taxi", "taxi", transform_taxi)
display(df_taxi_silver.limit(10))

### Drivers

In [0]:
def transform_drivers(df):
    """Clean Drivers data"""
    return df \
        .withColumn("FirstName", trim(col("FirstName"))) \
        .withColumn("LastName", trim(col("LastName"))) \
        .withColumn("Phone", regexp_replace(trim(col("Phone")), r"[^\d+]", "")) \
        .withColumn("Email", lower(trim(col("Email")))) \
        .withColumn("LicenseNumber", upper(trim(col("LicenseNumber")))) \
        .withColumn("Gender", upper(trim(col("Gender")))) \
        .withColumn("Age", col("Age").cast(IntegerType())) \
        .withColumn("YearsExperience", col("YearsExperience").cast(IntegerType())) \
        .withColumn("Rating", col("Rating").cast(DoubleType())) \
        .withColumn("JoinDate", to_date(col("JoinDate"))) \
        .na.fill({
            "FirstName": "Unknown",
            "LastName": "Unknown",
            "Phone": "",
            "Email": "",
            "LicenseNumber": "UNKNOWN",
            "Gender": "U",
            "Age": 30,
            "YearsExperience": 0,
            "Rating": 0.0
        })

df_drivers_silver = clean_and_save_to_silver("drivers", "drivers", transform_drivers)
display(df_drivers_silver.limit(10))

### Passengers

In [0]:
def transform_passengers(df):
    """
    Clean Passengers data with Singapore address standardization
    """
    return df \
        .withColumn("FirstName", trim(col("FirstName"))) \
        .withColumn("LastName", trim(col("LastName"))) \
        .withColumn("Phone", regexp_replace(trim(col("Phone")), r"[^\d+]", "")) \
        .withColumn("Email", lower(trim(col("Email")))) \
        .withColumn("MemberCategory", upper(trim(col("MemberCategory")))) \
        .withColumn("Gender", upper(trim(col("Gender")))) \
        .withColumn("Age", col("Age").cast(IntegerType())) \
        .withColumn("RegistrationDate", to_date(col("RegistrationDate"))) \
        .withColumn(
            "Address",
            # Standardize "Singapore" spelling
            regexp_replace(
                regexp_replace(
                    regexp_replace(
                        trim(col("Address")),
                        r"(?i)singapre", "Singapore"
                    ),
                    r"(?i)singapor\b", "Singapore"
                ),
                r"(?i)singpore", "Singapore"
            )
        ) \
        .withColumn(
            "Address",
            # Standardize block abbreviations
            regexp_replace(col("Address"), r"(?i)^block ", "Blk ")
        ) \
        .na.fill({
            "FirstName": "Unknown",
            "LastName": "Unknown",
            "Phone": "",
            "Email": "",
            "MemberCategory": "C",
            "Gender": "U",
            "Age": 30,
            "Address": "Unknown"
        })

df_passengers_silver = clean_and_save_to_silver("passengers", "passengers", transform_passengers)
display(df_passengers_silver.limit(10))

### Payment Type

In [0]:
def transform_payment_type(df):
    """Clean PaymentType data"""
    return df \
        .withColumn("TypeName", trim(col("TypeName"))) \
        .na.fill({"TypeName": "Unknown"})

df_payment_type_silver = clean_and_save_to_silver("payment_type", "payment_type", transform_payment_type)
display(df_payment_type_silver)

### Place (Location)

In [0]:
def transform_place(df):
    """
    Clean Place data with Singapore location standardization
    """
    return df \
        .withColumn(
            "PlaceName",
            # Standardize "Singapore" in place names
            regexp_replace(
                regexp_replace(
                    regexp_replace(
                        trim(col("PlaceName")),
                        r"(?i)singapre", "Singapore"
                    ),
                    r"(?i)singapor\b", "Singapore"
                ),
                r"(?i)singpore", "Singapore"
            )
        ) \
        .withColumn("Latitude", col("Latitude").cast(DoubleType())) \
        .withColumn("Longitude", col("Longitude").cast(DoubleType())) \
        .withColumn("PostalCode", col("PostalCode").cast(StringType())) \
        .na.fill({
            "PlaceName": "Unknown Location",
            "Latitude": 1.3521,
            "Longitude": 103.8198,
            "PostalCode": "000000"
        })

df_place_silver = clean_and_save_to_silver("place", "place", transform_place)
display(df_place_silver.limit(10))

## Transform Fact Tables

- **Booking Fact Table Transformation**
  - Cleans and standardizes booking data (timestamps, status, fare, distance).
  - Standardizes booking status values (Completed, Cancelled, In Progress).
  - Fills missing values with defaults.

- **Rewards Fact Table Transformation**
  - Cleans rewards data (points, earned date, description).
  - Fills missing values with defaults.

- **Data Quality Checks**
  - Checks for nulls in key columns for drivers, passengers, and booking tables.
  - Prints summary of null counts for each key column.

- **Transformation Summary**
  - Prints record counts and valid record counts for all silver tables.
  - Displays total records processed.

### Booking

In [0]:
def transform_booking(df):
    """Clean Booking fact table data"""
    return df \
        .withColumn("BookingDateTime", to_timestamp(col("BookingDateTime"))) \
        .withColumn("PickupTime", to_timestamp(col("PickupTime"))) \
        .withColumn("DropoffTime", to_timestamp(col("DropoffTime"))) \
        .withColumn("Distance", col("Distance").cast(DoubleType())) \
        .withColumn("Fare", col("Fare").cast(DoubleType())) \
        .withColumn("Status", trim(col("Status"))) \
        .withColumn(
            "Status",
            # Standardize status values
            when(col("Status").isin("Completed", "COMPLETED", "Complete"), "Completed")
            .when(col("Status").isin("Cancelled", "CANCELLED", "Canceled"), "Cancelled")
            .when(col("Status").isin("InProgress", "IN PROGRESS", "In Progress"), "In Progress")
            .otherwise(col("Status"))
        ) \
        .na.fill({
            "Distance": 0.0,
            "Fare": 0.0,
            "Status": "Unknown"
        })

df_booking_silver = clean_and_save_to_silver("booking", "booking", transform_booking)
display(df_booking_silver.limit(10))

### Rewards

In [0]:
def transform_rewards(df):
    """Clean Rewards data"""
    return df \
        .withColumn("Points", col("Points").cast(IntegerType())) \
        .withColumn("EarnedDate", to_date(col("EarnedDate"))) \
        .withColumn("Description", trim(col("Description"))) \
        .na.fill({
            "Points": 0,
            "Description": "No description"
        })

df_rewards_silver = clean_and_save_to_silver("rewards", "rewards", transform_rewards)
display(df_rewards_silver.limit(10))

## Data Quality Checks

In [0]:
# Check for Nulls in Key Columns
def check_nulls(table_name, key_columns):
    """Check for null values in key columns"""
    table_full = silver_table(table_name)
    df = spark.table(table_full)
    
    print(f"\n{'='*60}")
    print(f"NULL CHECK: {table_full}")
    print(f"{'='*60}")
    
    for col_name in key_columns:
        null_count = df.filter(col(col_name).isNull()).count()
        total_count = df.count()
        print(f"  {col_name}: {null_count} nulls out of {total_count} records")
    
    return df

# Check key tables
check_nulls("drivers", ["DriverID", "FirstName", "Email"])
check_nulls("passengers", ["PassengerID", "FirstName", "Email"])
check_nulls("booking", ["BookingID", "PassengerID", "DriverID", "TaxiID"])

## Summary

- Prints a transformation summary for all silver tables.
- For each table, shows total record count and count of valid records (where `data_quality_flag` is "VALID").
- Displays the total number of records processed across all tables.

In [0]:
print("="*80)
print("SILVER LAYER - TRANSFORMATION SUMMARY")
print("="*80)

silver_tables = [
    "taxi_type", "taxi", "drivers", "passengers", 
    "payment_type", "place", "booking", "rewards"
]

total_records = 0
for table in silver_tables:
    table_full = silver_table(table)
    try:
        df = spark.table(table_full)
        total = df.count()
        valid = df.filter(col("data_quality_flag") == "VALID").count()
        print(f"{table_full:<50} {total:>10,} records ({valid:,} valid)")
        total_records += total
    except Exception as e:
        print(f"{table_full:<50} {'ERROR':>10}")

print("-"*80)
print(f"{'TOTAL':<50} {total_records:>10,} records")
print("="*80)

## Show All Silver Tables

In [0]:
print(f"\nSilver Tables in {UC_CATALOG}.{UC_SCHEMA}:")
spark.sql(f"SHOW TABLES IN {UC_CATALOG}.{UC_SCHEMA}") \
    .filter("tableName LIKE 'silver_%'") \
    .select("database", "tableName", "isTemporary") \
    .show(truncate=False)

## End of Silver Layer Notebook

**Data Quality Improvements:**
- Standardized Singapore spelling in addresses and place names
- Cleaned phone numbers and emails
- Standardized status values
- Fixed data types
- Handled null values
- Added quality flags

**Next Steps:**
- Run the Gold Layer notebook (03_Gold_Layer_Star_Schema)
- Update Gold notebook to read from: `workspace.rebu.silver_{table}`